# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [1]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [6]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "test_20260530.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data_extended.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

Project root: c:\Projects\ProphetGP
Config loaded: {'data': {'reactant_column': 'Mol.1', 'target_column': ['Emission Peak', 'FWHM'], 'reactant_delimiter': '|', 'ignore_columns': ['PLQY', 'Pristine Emission', 'Pristine FWHM', 'Pristine PLQY'], 'reactant_allowed_values': [], 'condition_ranges': {'Temperature': {'min': 0.0, 'max': 600.0, 'allowed_values': None, 'grid_points': 7}}, 'explicit_condition_types': {'Temperature': 'continuous'}}, 'featurization': {'featuriser': 'topo_physchem', 'combine_strategy': 'concat'}, 'optimization': {'objective': 'target', 'suggestion_strategy': 'best_output', 'standardize_gp_inputs': True, 'standardize_gp_targets': True, 'target_value': None, 'target_objectives': {'Emission Peak': {'objective': 'target', 'target_value': 490.0, 'weight': 2.0}, 'FWHM': {'objective': 'minimize', 'target_value': None, 'weight': 1.0}}, 'n_restarts': 10, 'raw_samples': 128, 'target_search_size': 5000, 'condition_grid_points': 25, 'n_candidates': 3}}


In [7]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

Available featurisers count: 11
['bag_of_characters', 'drfp', 'ecfp_fingerprints', 'fragments', 'molecular_graphs', 'morgan_fp', 'mqn_features', 'one_hot', 'rdkit_descriptors', 'rxnfp', 'topo_physchem']


In [8]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])
print("molecular representation check:", np.unique(artifacts.x_train[:, :-1], axis=0).shape)
print("rank of training data:", np.linalg.matrix_rank(artifacts.x_train[:, :-1]))

Train rows: 67
Feature dims: 50
molecular representation check: (41, 49)
rank of training data: 34


In [9]:
# 3) 다음 실험 조건 후보 추천 (raw + 해석 결과)
# strategy: "best_output" | "best_information"
n_candidates = 3
strategy = "best_output"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
print("Decoded candidates:")
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(f"- candidate_{idx}")
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])
    print("  nearest_known_reactants_input:", row["nearest_known_reactants_input"])
    print("  Temperature:", row["Temperature"])
    print("  mapped input:", row)

# suggestions.raw_candidates

Strategy: best_output
Raw candidates shape: (3, 50)
Decoded candidates:
- candidate_1
  predicted mean: {'Emission Peak': 488.4761265975226, 'FWHM': 71.11853488483055}
  predicted std: {'Emission Peak': 21.120513331302057, 'FWHM': 11.673618105334148}
  target gap: {'Emission Peak': 1.5238734024774203, 'FWHM': None}
  nearest_known_reactants_input: 1,5-Diaminonaphthalene
  Temperature: 180.0
  mapped input: {'predicted_target_mean': {'Emission Peak': 488.4761265975226, 'FWHM': 71.11853488483055}, 'predicted_target_std': {'Emission Peak': 21.120513331302057, 'FWHM': 11.673618105334148}, 'target_gap': {'Emission Peak': 1.5238734024774203, 'FWHM': None}, 'objective_score': -74.1662816897854, 'information_score': 53.91464476793826, 'total_score': -74.1662816897854, 'ranking_strategy': 'best_output', 'mapped_reactants_input': '1,5-Diaminonaphthalene', 'mapped_reactants_smiles': ['Nc1cccc2c(N)cccc12'], 'nearest_known_reactants_input': '1,5-Diaminonaphthalene', 'nearest_known_reactants_smiles'

In [5]:
# 3b) 지정 입력에 대한 예측 (GP posterior mean / std)
# suggest_next_experiments와 달리, 사용자가 정한 반응물·조건에 대한 예측값을 조회한다.
query_inputs = [
    {"reactants": "2,6-Diaminonaphthalene", "Temperature": 200.0},
    {"reactants": "5-amino-1,10-phenanthroline|salicylic acid", "Temperature": 250.0},
]
prediction_result = pipeline.predict_targets(artifacts, query_inputs)

for idx, row in enumerate(prediction_result.predictions, 1):
    print(f"- query_{idx}")
    print("  reactants:", row["reactants_input"])
    print("  conditions:", row["conditions"])
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])

- query_1
  reactants: 2,6-Diaminonaphthalene
  conditions: {'Temperature': 200.0}
  predicted mean: {'Emission Peak': 360.2760796985147, 'FWHM': 74.32547439672341}
  predicted std: {'Emission Peak': 2.980865047721137, 'FWHM': 9.462737190318293}
  target gap: {'Emission Peak': 129.7239203014853, 'FWHM': None}
- query_2
  reactants: 5-amino-1,10-phenanthroline|salicylic acid
  conditions: {'Temperature': 250.0}
  predicted mean: {'Emission Peak': 254.97553664567812, 'FWHM': 89.07241605628063}
  predicted std: {'Emission Peak': 4.461124147004263, 'FWHM': 11.103498721353796}
  target gap: {'Emission Peak': 235.02446335432188, 'FWHM': None}


In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)